# Fish larvae — interactive map

Loads `data/fish.csv` from this repo (run Jupyter with the project folder as the working directory).

**Note on `other_fish.csv`:** That file is ichthyoplankton **egg counts** from a 2022 cruise (tow segments with salinity/temperature). It overlaps **geographically** with the California Current / Southern California Bight (similar lat/lon range as `fish.csv`), but it does **not** list species-level larvae IDs in the same way as `fish.csv`, and the `other_fish_eggs` column is an aggregate “other” category—not comparable species rows. For a species-level larvae dashboard, keep the main map on `fish.csv` only; use `other_fish.csv` later only if you add a separate “egg survey” or environmental layer, not merged as the same points.


In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

# Project root = parent of this notebook's folder if you use notebooks/; adjust if needed
DATA_PATH = Path("data/fish.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/fish.csv")

fish_df = pd.read_csv(DATA_PATH)
fish_df.head()


In [ ]:
# One row per species observation at a tow; same lat/lon appears for multiple species
points = gpd.GeoDataFrame(
    fish_df,
    geometry=gpd.points_from_xy(fish_df["longitude"], fish_df["latitude"]),
    crs="EPSG:4326",
)
points.head()


In [ ]:
# Aggregate all species at each exact coordinate for one circle per station
sorted_gdf = points[
    ["common_name", "scientific_name", "time", "latitude", "longitude", "larvae_count", "larvae_10m2", "geometry"]
].sort_values(by="larvae_count", ascending=False)

grouped = (
    sorted_gdf.groupby("geometry", as_index=False)
    .agg(
        obs_count=("geometry", "size"),
        total_larvae=("larvae_count", "sum"),
        larvae_counts=("larvae_count", list),
        larvae_10m2_values=("larvae_10m2", list),
        common_names=("common_name", list),
        scientific_names=("scientific_name", list),
        times=("time", list),
    )
)
grouped = gpd.GeoDataFrame(grouped, geometry="geometry", crs=sorted_gdf.crs)

def fmt_hover(row):
    parts = []
    for c, s, lc in zip(row["common_names"], row["scientific_names"], row["larvae_counts"]):
        c = c if pd.notna(c) else "—"
        s = s if pd.notna(s) else "—"
        parts.append(f"<b>{c}</b><br><i>{s}</i><br>larvae: {lc}")
    return "<hr>".join(parts)

grouped["hover_html"] = grouped.apply(fmt_hover, axis=1)

def fmt_tooltip_plain(row):
    pairs = []
    for c, s in zip(row["common_names"], row["scientific_names"]):
        c = c if pd.notna(c) else "—"
        s = s if pd.notna(s) else "—"
        pairs.append(f"{c} | {s}")
    return " • ".join(pairs)[:400]

grouped["tooltip_plain"] = grouped.apply(fmt_tooltip_plain, axis=1)

grouped.head()


In [ ]:
# Interactive map (pan/zoom = draggable map). Tooltip = hover; popup = click.
m = grouped.explore(
    column="total_larvae",
    cmap="YlOrRd",
    legend=True,
    tooltip=["tooltip_plain"],
    popup=["hover_html"],
    marker_kwds={"radius": 8},
    tiles="OpenStreetMap",
    style_kwds={"stroke": True, "color": "#333", "weight": 1, "fillOpacity": 0.75},
)
# Folium renders in notebook
m
